# NeuraMind Workshop — Phase 1: The Perceptron Deep Dive
### Architecture, Vector Dimensions, Mathematics, Scratch NumPy Implementation, and Convergence Dynamics

Welcome to the **NeuraMind Deep Learning Workshop**. In this notebook, we explore the foundational atomic computational unit of neural networks: the **Perceptron**.

---

## 1. Mathematical Formulation & Vector Dimensions

A Perceptron computes a linear combination of its input features, offsets it by a scalar bias, and applies a non-linear threshold activation function:

$$\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_D \end{bmatrix} \in \mathbb{R}^{D \times 1}, \quad \mathbf{w} = \begin{bmatrix} w_1 \\ w_2 \\ \vdots \\ w_D \end{bmatrix} \in \mathbb{R}^{D \times 1}, \quad b \in \mathbb{R}$$

The linear combination (scalar dot product) is:
$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{j=1}^D w_j x_j + b$$

The Heaviside step activation function yields binary classification predictions:
$$\hat{y} = f(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

### Compact Matrix Form (Augmented Input & Weight Vectors)
By prepending a constant $1$ to the feature vector and treating the bias $b$ as weight $w_0$:
$$\tilde{\mathbf{x}} = [1, x_1, x_2, \dots, x_D]^T, \quad \tilde{\mathbf{w}} = [b, w_1, w_2, \dots, w_D]^T \implies z = \tilde{\mathbf{w}}^T \tilde{\mathbf{x}}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic plotting styles
sns.set_theme(style="darkgrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
print("NumPy & Visualisation libraries imported successfully!")

### Concrete Numerical Walkthrough (Manual Calculation)

Consider a test applicant with:
- $\text{IQ} = 100$ ($x_1$)
- $\text{CGPA} = 5.1$ ($x_2$)
- Candidate model weights: $w_1 = 1.0$, $w_2 = 2.0$, and scalar bias $b = -105.0$

**1. Linear Dot Product ($z$):**
$$z = b + w_1 x_1 + w_2 x_2 = -105.0 + (1.0 \times 100) + (2.0 \times 5.1) = -105.0 + 100.0 + 10.2 = +5.2$$

**2. Activation Function (Heaviside Step):**
$$\hat{y} = \mathrm{Step}(5.2) = 1 \quad (\mathbf{Placed!})$$

> **Feature Importance Takeaway:** Notice that $w_{\text{CGPA}} = 2.0$ while $w_{\text{IQ}} = 1.0$. A 1-point increase in CGPA increases $z$ by $2.0$, twice as much as a 1-point increase in IQ. The weights directly encode the relative importance of each feature in the decision boundary!

In [ ]:
# Manual calculation verification in Python
x_test = np.array([100.0, 5.1])
w_test = np.array([1.0, 2.0])
b_test = -105.0

z_calc = np.dot(w_test, x_test) + b_test
y_hat_calc = 1 if z_calc >= 0 else 0

print(f"Computed z: {z_calc:.2f}")
print(f"Prediction y_hat: {y_hat_calc} -> {'Placed' if y_hat_calc == 1 else 'Not Placed'}")
assert y_hat_calc == 1, 'Calculation mismatch!'

## 2. Geometric Interpretation: Decision Boundary & Half-Spaces

Setting $z = 0$ defines the decision boundary:
$$w_1 x_1 + w_2 x_2 + b = 0$$

Solving for $x_2$ (the vertical axis):
$$x_2 = -\left(\frac{w_1}{w_2}\right) x_1 - \left(\frac{b}{w_2}\right)$$

- **Slope ($m$):** $-\frac{w_1}{w_2}$ (governed by feature weights)
- **Vertical Intercept ($c$):** $-\frac{b}{w_2}$ (governed by bias term)
- **Positive Half-Space ($z \ge 0$):** Points predicted as Class 1
- **Negative Half-Space ($z < 0$):** Points predicted as Class 0

## 3. The Unified Perceptron Update Rule

When a training sample $(\mathbf{x}_i, y_i)$ is evaluated:
$$\mathbf{w}_{\text{new}} = \mathbf{w}_{\text{old}} + \eta \,(y_i - \hat{y}_i)\,\mathbf{x}_i$$
$$b_{\text{new}} = b_{\text{old}} + \eta \,(y_i - \hat{y}_i)$$

| Scenario | True $y_i$ | Predicted $\hat{y}_i$ | Error $(y_i - \hat{y}_i)$ | Action Taken |
| :--- | :---: | :---: | :---: | :--- |
| **Correct** | $1$ or $0$ | $1$ or $0$ | $0$ | No update (line remains static) |
| **False Negative** | $1$ | $0$ | $+1$ | Adds $+\eta \mathbf{x}_i$ (pulls line toward point) |
| **False Positive** | $0$ | $1$ | $-1$ | Subtracts $-\eta \mathbf{x}_i$ (pushes line away) |

In [ ]:
class ScratchPerceptron:
    """
    Pure NumPy Implementation of the Classical Rosenblatt Perceptron.
    """
    def __init__(self, learning_rate=0.05, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.error_history = []
        self.weight_history = []
        
    def _step(self, z):
        return np.where(z >= 0, 1, 0)
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.error_history = []
        self.weight_history = []
        
        for epoch in range(self.epochs):
            misclassified = 0
            for i in range(n_samples):
                x_i = X[i]
                y_i = y[i]
                
                # 1. Forward pass (linear dot product)
                z = np.dot(self.weights, x_i) + self.bias
                y_hat = self._step(z)
                
                # 2. Error calculation
                error = y_i - y_hat
                
                # 3. Unified parameter update
                if error != 0:
                    self.weights += self.lr * error * x_i
                    self.bias += self.lr * error
                    misclassified += 1
                    
            self.error_history.append(misclassified)
            self.weight_history.append((self.weights.copy(), self.bias))
            
            # Early stopping if linearly separated
            if misclassified == 0:
                print(f"✓ Converged to 0 errors at Epoch {epoch+1}!")
                break
                
        return self
        
    def predict(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._step(z)

print("ScratchPerceptron class defined successfully!")

## 4. Experiment 1: Synthetic Placement Dataset (IQ & CGPA)

Let us generate a linearly separable 2D dataset simulating student placement outcomes based on **IQ** ($X_1$) and **CGPA** ($X_2$).

In [ ]:
from sklearn.datasets import make_blobs

np.random.seed(42)
# Generate 100 student instances across 2 clusters
X, y = make_blobs(n_samples=100, n_features=2, centers=[[2.5, 3.0], [6.5, 7.5]], cluster_std=1.0, random_state=42)

# Instantiate and fit our Scratch Perceptron
model = ScratchPerceptron(learning_rate=0.05, epochs=50)
model.fit(X, y)

w1, w2 = model.weights
b = model.bias
print(f"\nLearned Parameters:")
print(f"  Weight w1 (IQ):       {w1:.4f}")
print(f"  Weight w2 (CGPA):     {w2:.4f}")
print(f"  Bias b:               {b:.4f}")
print(f"  Slope (m = -w1/w2):   {-w1/w2:.4f}")
print(f"  Intercept (c = -b/w2):{-b/w2:.4f}")

## 5. Visualizing the Learned Decision Boundary & Generalization Regions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Create dense coordinate mesh for background contour
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid_pts = np.c_[xx.ravel(), yy.ravel()]
Z = model.predict(grid_pts).reshape(xx.shape)

# Plot decision regions
ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")

# Plot scatter points
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="black", s=70, linewidth=1.2)

# Plot explicit decision line: w1*x1 + w2*x2 + b = 0 => x2 = -(w1/w2)*x1 - (b/w2)
x_vals = np.array([x_min, x_max])
y_vals = -(w1 / w2) * x_vals - (b / w2)
ax.plot(x_vals, y_vals, color="#00F0FF", linewidth=3, label=f"Decision Boundary: {w1:.2f}x1 + {w2:.2f}x2 + {b:.2f} = 0")

ax.set_title("Perceptron Linear Decision Boundary & Generalization Mesh", fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Feature 1: IQ ($x_1$)", fontsize=13)
ax.set_ylabel("Feature 2: CGPA ($x_2$)", fontsize=13)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.legend(loc="upper left", frameon=True)
plt.show()

## 6. Training Curve: Error Convergence Across Epochs

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(model.error_history) + 1), model.error_history, marker='o', color="#EF4444", linewidth=2.5, markersize=7)
plt.title("Training Convergence: Misclassifications per Epoch", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Training Epoch", fontsize=12)
plt.ylabel("Number of Misclassified Samples", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## 7. The Fundamental Limitation: Linear Separability & The XOR Failure

Let us test the Perceptron on the classic Boolean logic gates: **AND**, **OR**, and **XOR**.

In [ ]:
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and   = np.array([0, 0, 0, 1])
y_or    = np.array([0, 1, 1, 1])
y_xor   = np.array([0, 1, 1, 0])

print("Training Perceptron on AND Gate:")
p_and = ScratchPerceptron(learning_rate=0.1, epochs=50).fit(X_logic, y_and)

print("\nTraining Perceptron on OR Gate:")
p_or = ScratchPerceptron(learning_rate=0.1, epochs=50).fit(X_logic, y_or)

print("\nTraining Perceptron on XOR Gate:")
p_xor = ScratchPerceptron(learning_rate=0.1, epochs=50).fit(X_logic, y_xor)

print(f"\nXOR Final Misclassifications: {p_xor.error_history[-1]} errors remaining!")
print("-> Conclusion: A single straight line CANNOT separate XOR. Multi-Layer Perceptrons (MLPs) are required!")